# 구조물 안전성 예측 — V2 개선 버전
**적용 기법:**
- A. 백본: EfficientNetV2-M (B5 → V2-M 업그레이드)
- B. 손실: Focal Loss + Label Smoothing 혼합
- C. 풀링: GeM Pooling (avgpool 대체)
- D. 데이터: Pseudo Labeling (2단계 학습)
- 기존 유지: Cross-View Attention · 5-Fold · Cosine Annealing w/ Warmup · TTA · AMP · EarlyStopping

## 1. 라이브러리 및 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import copy
import math
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    efficientnet_v2_m, EfficientNet_V2_M_Weights
)

from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

# ── 하이퍼파라미터 ──────────────────────────────────────────────────
CFG = {
    # [A] EfficientNetV2-M 권장 입력 크기
    'IMG_SIZE'        : 480,
    'EPOCHS'          : 30,
    'LEARNING_RATE'   : 1e-4,
    'BATCH_SIZE'      : 8,
    'SEED'            : 42,
    'N_FOLDS'         : 5,
    # [B] Focal Loss 파라미터
    'LABEL_SMOOTH'    : 0.05,
    'FOCAL_GAMMA'     : 1.5,   # 어려운 샘플 집중 강도 (1.0~2.0 권장)
    'FOCAL_ALPHA'     : 0.75,  # unstable 클래스 가중치 (불균형 시 조정)
    'TTA_STEPS'       : 4,
    'PATIENCE'        : 5,
    # [D] Pseudo Labeling 신뢰도 임계값
    'PSEUDO_THRESHOLD': 0.05,  # prob < 0.05 or prob > 0.95 인 샘플만 사용
    'BASE_PATH'       : '/content/drive/MyDrive/MyDrive/Stability_Project',
    'CKPT_DIR'        : '/content/drive/MyDrive/MyDrive/Stability_Project/checkpoints_v2',
    'CKPT_DIR_PL'     : '/content/drive/MyDrive/MyDrive/Stability_Project/checkpoints_v2_pl',
}

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['SEED'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. 데이터 압축 해제 및 로드

In [ ]:
BASE_PATH = CFG['BASE_PATH']
os.makedirs(CFG['CKPT_DIR'],    exist_ok=True)
os.makedirs(CFG['CKPT_DIR_PL'], exist_ok=True)

!mkdir -p /content/train_images /content/dev_images /content/test_images

for split, dest in [('train', '/content/train_images'),
                    ('dev',   '/content/dev_images'),
                    ('test',  '/content/test_images')]:
    zip_path = os.path.join(BASE_PATH, f'{split}.zip')
    if os.path.exists(zip_path):
        os.system(f'unzip -q "{zip_path}" -d {dest}')
        print(f'✅ {split}.zip 압축 해제 완료')
    else:
        print(f'❌ {zip_path} 없음')

In [ ]:
train_df = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))
val_df   = pd.read_csv(os.path.join(BASE_PATH, 'dev.csv'))
test_df  = pd.read_csv(os.path.join(BASE_PATH, 'sample_submission.csv'))

train_df['root_dir'] = '/content/train_images/train'
val_df['root_dir']   = '/content/dev_images/dev'
test_df['root_dir']  = '/content/test_images/test'

all_df = pd.concat([train_df, val_df], ignore_index=True)

print(f'전체 학습 데이터: {len(all_df)}  |  테스트: {len(test_df)}')
print(all_df['label'].value_counts())

# 클래스 비율 확인 → FOCAL_ALPHA 조정 참고용
unstable_ratio = (all_df['label'] == 'unstable').mean()
print(f'\nunstable 비율: {unstable_ratio:.3f}')
print('※ unstable 비율이 0.3 이하면 FOCAL_ALPHA를 0.8~0.9로 올리세요')

## 3. 데이터셋 및 Augmentation

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
])

val_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

tta_transforms = [
    val_transform,
    transforms.Compose([
        transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((int(CFG['IMG_SIZE'] * 1.1), int(CFG['IMG_SIZE'] * 1.1))),
        transforms.CenterCrop(CFG['IMG_SIZE']),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
]

In [ ]:
class MultiViewDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.is_test   = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def _load_img(self, path):
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        sample_id = str(row['id'])
        root_dir  = row.get('root_dir', '/content/test_images/test')
        folder    = os.path.join(root_dir, sample_id)

        views = [
            self._load_img(os.path.join(folder, 'front.png')),
            self._load_img(os.path.join(folder, 'top.png')),
        ]

        if self.is_test:
            return views

        label = self.label_map[row['label']]
        return views, label

## 4. 모델 정의
### [A] EfficientNetV2-M 백본  
### [C] GeM Pooling  
### Cross-View Attention 융합 (유지)

In [ ]:
# ── [C] GeM Pooling ────────────────────────────────────────────────
class GeM(nn.Module):
    """
    Generalized Mean Pooling.
    - p=1 → 평균 풀링과 동일
    - p→∞ → 최대 풀링에 근접
    - p를 학습 파라미터로 두면 태스크에 맞게 자동 최적화됨
    검색/분류 태스크에서 avgpool 대비 일관된 성능 향상 보고됨
    """
    def __init__(self, p: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.adaptive_avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            output_size=1
        ).pow(1.0 / self.p)


# ── Cross-View Attention (기존 유지) ───────────────────────────────
class CrossViewAttention(nn.Module):
    """
    front ↔ top 양방향 attention 융합.
    f1(front)이 f2(top)를 참조하고, f2가 f1을 참조.
    Residual + LayerNorm으로 학습 안정성 확보.
    """
    def __init__(self, feat_dim: int):
        super().__init__()
        self.scale = math.sqrt(feat_dim)

        self.q1 = nn.Linear(feat_dim, feat_dim, bias=False)
        self.k2 = nn.Linear(feat_dim, feat_dim, bias=False)
        self.v2 = nn.Linear(feat_dim, feat_dim, bias=False)

        self.q2 = nn.Linear(feat_dim, feat_dim, bias=False)
        self.k1 = nn.Linear(feat_dim, feat_dim, bias=False)
        self.v1 = nn.Linear(feat_dim, feat_dim, bias=False)

        self.norm1 = nn.LayerNorm(feat_dim)
        self.norm2 = nn.LayerNorm(feat_dim)

    def forward(self, f1, f2):
        attn_12 = torch.sigmoid((self.q1(f1) * self.k2(f2)) / self.scale)
        out1    = self.norm1(f1 + attn_12 * self.v2(f2))

        attn_21 = torch.sigmoid((self.q2(f2) * self.k1(f1)) / self.scale)
        out2    = self.norm2(f2 + attn_21 * self.v1(f1))

        return torch.cat([out1, out2], dim=1)  # (B, feat_dim * 2)


# ── [A] EfficientNetV2-M + [C] GeM + Cross-View Attention ─────────
class MultiViewEfficientNetV2(nn.Module):
    """
    EfficientNetV2-M 백본 + GeM Pooling + CrossViewAttention.
    EfficientNetV2-M: B5보다 학습 속도 빠르고 정확도 높음.
    feature dim: 1280 (V2-M 기본값)
    """
    def __init__(self, num_classes: int = 1, dropout: float = 0.4):
        super().__init__()
        backbone = efficientnet_v2_m(weights=EfficientNet_V2_M_Weights.DEFAULT)

        # features만 추출 (classifier head 제거)
        self.features = backbone.features
        self.pool     = GeM(p=3.0)          # [C] avgpool → GeM
        feat_dim      = 1280                 # EfficientNetV2-M output dim

        self.fusion = CrossViewAttention(feat_dim)

        self.classifier = nn.Sequential(
            nn.Linear(feat_dim * 2, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(128, num_classes),
        )

    def extract(self, x):
        return self.pool(self.features(x)).flatten(1)  # (B, 1280)

    def forward(self, views):
        f1    = self.extract(views[0])
        f2    = self.extract(views[1])
        fused = self.fusion(f1, f2)         # (B, 2560)
        return self.classifier(fused)       # (B, 1)


# 동작 확인
dummy = MultiViewEfficientNetV2().to(device)
dummy_v = [torch.randn(2, 3, 480, 480).to(device)] * 2
print(f'Output shape: {dummy(dummy_v).shape}')  # (2, 1)
del dummy, dummy_v

## 5. 손실 함수 및 학습 유틸
### [B] Focal Loss + Label Smoothing

In [ ]:
# ── [B] Focal Loss + Label Smoothing ──────────────────────────────
class FocalLabelSmoothingBCE(nn.Module):
    """
    Focal Loss + Label Smoothing 혼합 손실 함수.

    Focal Loss 효과:
      - 쉬운 샘플(모델이 이미 잘 맞추는 것)의 loss를 낮춤
      - 어려운 샘플에 학습 집중 → 결정 경계 근처 샘플 개선
      - gamma=0이면 일반 BCE와 동일

    alpha 가이드:
      - unstable 비율 ~0.5 → alpha=0.75
      - unstable 비율 ~0.3 → alpha=0.80
      - unstable 비율 ~0.2 → alpha=0.85
    """
    def __init__(self,
                 smoothing: float = 0.05,
                 gamma: float = 1.5,
                 alpha: float = 0.75):
        super().__init__()
        self.smoothing = smoothing
        self.gamma     = gamma
        self.alpha     = alpha

    def forward(self, logits, targets):
        # Label Smoothing 적용
        smooth_targets = targets * (1.0 - self.smoothing) + self.smoothing * 0.5

        # BCE (reduction='none' → 샘플별 loss 계산)
        bce = F.binary_cross_entropy_with_logits(
            logits, smooth_targets, reduction='none'
        )

        # Focal weight: (1 - p_t)^gamma
        probs  = torch.sigmoid(logits)
        p_t    = probs * targets + (1 - probs) * (1 - targets)
        focal  = (1.0 - p_t) ** self.gamma

        # Class-balancing alpha weight
        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)

        loss = alpha_t * focal * bce
        return loss.mean()


def compute_logloss(probs, labels):
    """대회 공식 Binary Log-Loss."""
    eps = 1e-15
    p = np.clip(probs, eps, 1 - eps)
    return -np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = 0.0

    for views, labels in tqdm(loader, desc='Train', leave=False):
        views  = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(views).view(-1)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []

    for views, labels in tqdm(loader, desc='Val', leave=False):
        views = [v.to(device) for v in views]
        with torch.cuda.amp.autocast():
            logits = model(views).view(-1)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

    all_probs  = np.array(all_probs,  dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    return compute_logloss(all_probs, all_labels), np.mean((all_probs > 0.5) == all_labels)


def run_fold_training(train_df_fold, val_df_fold, fold, ckpt_dir):
    """
    단일 Fold 학습 함수. Phase1 / Phase2(Pseudo) 공통으로 사용.
    체크포인트 존재 시 스킵. best val logloss 반환.
    """
    ckpt_path = os.path.join(ckpt_dir, f'fold{fold}_best.pth')

    val_ds     = MultiViewDataset(val_df_fold, transform=val_transform)
    val_loader = DataLoader(val_ds, batch_size=CFG['BATCH_SIZE'],
                            shuffle=False, num_workers=2, pin_memory=True)

    # 체크포인트 존재 → 스킵
    if os.path.exists(ckpt_path):
        print(f'  체크포인트 존재 → 학습 스킵')
        model = MultiViewEfficientNetV2().to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        val_logloss, val_acc = validate(model, val_loader, device)
        print(f'  Fold {fold} val logloss: {val_logloss:.5f}  acc: {val_acc:.4f}')
        return val_logloss

    train_ds     = MultiViewDataset(train_df_fold, transform=train_transform)
    train_loader = DataLoader(train_ds, batch_size=CFG['BATCH_SIZE'],
                              shuffle=True, num_workers=2, pin_memory=True)

    model     = MultiViewEfficientNetV2().to(device)
    criterion = FocalLabelSmoothingBCE(
        smoothing=CFG['LABEL_SMOOTH'],
        gamma=CFG['FOCAL_GAMMA'],
        alpha=CFG['FOCAL_ALPHA'],
    )

    # Differential LR: backbone 낮게, head 높게
    backbone_params = list(model.features.parameters()) + list(model.pool.parameters())
    head_params     = list(model.fusion.parameters()) + list(model.classifier.parameters())
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': CFG['LEARNING_RATE'] * 0.1},
        {'params': head_params,     'lr': CFG['LEARNING_RATE']},
    ], weight_decay=1e-4)

    total_steps  = CFG['EPOCHS'] * len(train_loader)
    warmup_steps = 2 * len(train_loader)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = torch.cuda.amp.GradScaler()

    best_logloss = float('inf')
    patience_cnt = 0
    global_step  = 0

    for epoch in range(1, CFG['EPOCHS'] + 1):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device
        )
        val_logloss, val_acc = validate(model, val_loader, device)

        # batch 단위 scheduler step
        for _ in range(len(train_loader)):
            scheduler.step()
        global_step += len(train_loader)

        cur_lr = optimizer.param_groups[-1]['lr']
        print(f'  Epoch [{epoch:02d}/{CFG["EPOCHS"]}] '
              f'train={train_loss:.4f}  '
              f'val_logloss={val_logloss:.5f}  '
              f'acc={val_acc:.4f}  '
              f'lr={cur_lr:.2e}')

        if val_logloss < best_logloss:
            best_logloss = val_logloss
            torch.save(model.state_dict(), ckpt_path)
            patience_cnt = 0
            print(f'  ✅ Best updated → {best_logloss:.5f}')
        else:
            patience_cnt += 1
            if patience_cnt >= CFG['PATIENCE']:
                print(f'  ⏹ Early stopping at epoch {epoch}')
                break

    return best_logloss

## 6. TTA 추론 유틸

In [ ]:
class TTADataset(Dataset):
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        sample_id = str(row['id'])
        root_dir  = row.get('root_dir', '/content/test_images/test')
        folder    = os.path.join(root_dir, sample_id)
        return [
            self.transform(Image.open(os.path.join(folder, 'front.png')).convert('RGB')),
            self.transform(Image.open(os.path.join(folder, 'top.png')).convert('RGB')),
        ]


@torch.no_grad()
def predict_tta(model, df, device, tta_tfms):
    """여러 TTA 변환 평균 예측 → 1D numpy array 반환."""
    model.eval()
    all_tta = []

    for tfm in tta_tfms:
        ds     = TTADataset(df, tfm)
        loader = DataLoader(ds, batch_size=CFG['BATCH_SIZE'],
                            shuffle=False, num_workers=2)
        probs  = []
        for views in tqdm(loader, desc='TTA', leave=False):
            views = [v.to(device) for v in views]
            with torch.cuda.amp.autocast():
                logits = model(views).view(-1)
            probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_tta.append(np.array(probs))

    return np.mean(all_tta, axis=0)


def ensemble_predict(df, ckpt_dir, device, tta_tfms, n_folds):
    """n_folds 모델 TTA 예측 평균 앙상블."""
    fold_probs = []
    for fold in range(1, n_folds + 1):
        print(f'  Fold {fold} TTA 추론...')
        model = MultiViewEfficientNetV2().to(device)
        model.load_state_dict(
            torch.load(os.path.join(ckpt_dir, f'fold{fold}_best.pth'),
                       map_location=device)
        )
        probs = predict_tta(model, df, device, tta_tfms)
        fold_probs.append(probs)
        print(f'    done. mean_prob={probs.mean():.4f}')
    return np.mean(fold_probs, axis=0)

## 7. Phase 1 — 5-Fold 학습 (원본 데이터)

In [ ]:
skf = StratifiedKFold(
    n_splits=CFG['N_FOLDS'],
    shuffle=True,
    random_state=CFG['SEED']
)

print('=' * 55)
print('  Phase 1: 원본 데이터 5-Fold 학습')
print('=' * 55)

phase1_scores = []

for fold, (train_idx, val_idx) in enumerate(
        skf.split(all_df, all_df['label']), start=1):

    print(f'\n{"-"*55}')
    print(f'  FOLD {fold} / {CFG["N_FOLDS"]}')
    print(f'{"-"*55}')

    fold_train = all_df.iloc[train_idx].reset_index(drop=True)
    fold_val   = all_df.iloc[val_idx].reset_index(drop=True)

    score = run_fold_training(fold_train, fold_val, fold, CFG['CKPT_DIR'])
    phase1_scores.append(score)

print(f'\n{"="*55}')
print(f'  Phase 1 CV Mean Log-Loss: {np.mean(phase1_scores):.5f}')
print(f'  Per-fold: {[f"{s:.5f}" for s in phase1_scores]}')

## 8. Phase 1 TTA 추론 → Pseudo Label 생성
### [D] Pseudo Labeling

In [ ]:
print('Phase 1 앙상블 추론 (test 데이터 → Pseudo Label 생성)...')

phase1_probs = ensemble_predict(
    test_df, CFG['CKPT_DIR'], device, tta_transforms, CFG['N_FOLDS']
)

# ── [D] 신뢰도 높은 샘플만 Pseudo Label로 선택 ─────────────────────
thr = CFG['PSEUDO_THRESHOLD']
confident_mask = (phase1_probs < thr) | (phase1_probs > 1 - thr)

pseudo_df = test_df.copy()
pseudo_df['prob']  = phase1_probs
pseudo_df['label'] = np.where(phase1_probs >= 0.5, 'unstable', 'stable')
# root_dir는 이미 test_df에 있음

pseudo_selected = pseudo_df[confident_mask].reset_index(drop=True)

print(f'\nPseudo Label 생성 완료')
print(f'  전체 test: {len(test_df)}개')
print(f'  신뢰도 통과 (prob < {thr} or > {1-thr}): {len(pseudo_selected)}개')
print(f'  비율: {len(pseudo_selected)/len(test_df)*100:.1f}%')
print(f'  pseudo label 분포:\n{pseudo_selected["label"].value_counts()}')

# 신뢰도 통과 샘플이 너무 적으면 임계값 완화 안내
if len(pseudo_selected) < 50:
    print('\n⚠️  Pseudo 샘플이 50개 미만입니다.')
    print(f'   PSEUDO_THRESHOLD를 0.1~0.15로 올려보세요.')

## 9. Phase 2 — 5-Fold 학습 (원본 + Pseudo Label 합산)

In [ ]:
# 원본 학습 데이터 + Pseudo Label 합산
augmented_df = pd.concat(
    [all_df, pseudo_selected[['id', 'label', 'root_dir']]],
    ignore_index=True
)

print('=' * 55)
print('  Phase 2: 원본 + Pseudo Label 5-Fold 학습')
print(f'  학습 데이터: {len(all_df)} → {len(augmented_df)}개')
print('=' * 55)

phase2_scores = []

# Phase 2는 val을 원본 all_df 기준으로만 구성 (pseudo 샘플은 학습에만 사용)
for fold, (train_idx, val_idx) in enumerate(
        skf.split(all_df, all_df['label']), start=1):

    print(f'\n{"-"*55}')
    print(f'  FOLD {fold} / {CFG["N_FOLDS"]}')
    print(f'{"-"*55}')

    # val: 원본 all_df의 val_idx만 사용
    fold_val = all_df.iloc[val_idx].reset_index(drop=True)

    # train: 원본 train_idx + pseudo 전체
    fold_train_orig  = all_df.iloc[train_idx].reset_index(drop=True)
    fold_train_phase2 = pd.concat(
        [fold_train_orig, pseudo_selected[['id', 'label', 'root_dir']]],
        ignore_index=True
    )

    score = run_fold_training(
        fold_train_phase2, fold_val, fold, CFG['CKPT_DIR_PL']
    )
    phase2_scores.append(score)

print(f'\n{"="*55}')
print(f'  Phase 1 CV Mean: {np.mean(phase1_scores):.5f}')
print(f'  Phase 2 CV Mean: {np.mean(phase2_scores):.5f}')
improvement = np.mean(phase1_scores) - np.mean(phase2_scores)
print(f'  개선량: {improvement:+.5f} ({"향상" if improvement > 0 else "저하"})')

## 10. 최종 추론 및 제출 파일 생성
Phase 2가 Phase 1보다 좋으면 Phase 2 모델로, 아니면 Phase 1 모델로 제출

In [ ]:
# Phase 2가 개선되었으면 Phase 2 체크포인트 사용, 아니면 Phase 1
if np.mean(phase2_scores) < np.mean(phase1_scores):
    best_ckpt_dir = CFG['CKPT_DIR_PL']
    print(f'Phase 2 채택 (logloss: {np.mean(phase2_scores):.5f})')
else:
    best_ckpt_dir = CFG['CKPT_DIR']
    print(f'Phase 1 채택 (logloss: {np.mean(phase1_scores):.5f})')
    print('※ Pseudo Label이 도움이 되지 않았습니다. PSEUDO_THRESHOLD 조정을 권장합니다.')

print('\n최종 TTA 앙상블 추론 중...')
final_probs = ensemble_predict(
    test_df, best_ckpt_dir, device, tta_transforms, CFG['N_FOLDS']
)

submission = pd.DataFrame({
    'id'           : test_df['id'],
    'unstable_prob': final_probs,
    'stable_prob'  : 1.0 - final_probs,
})

submission.to_csv('submission.csv', index=False, encoding='UTF-8-sig')
print('\nsubmission.csv 저장 완료.')
print(submission.head(10))

## 부록 — 변경 사항 요약

| 항목 | V1 | V2 (이번 버전) |
|---|---|---|
| **[A] 백본** | EfficientNet-B5 (2048-dim) | EfficientNetV2-M (1280-dim, 더 빠름) |
| **[B] 손실** | LabelSmoothing BCE | Focal + LabelSmoothing BCE |
| **[C] 풀링** | AdaptiveAvgPool | GeM Pooling (학습 가능한 p) |
| **[D] 데이터** | train+dev만 사용 | Pseudo Labeling (test 고신뢰 샘플 추가) |
| 학습 단계 | 1단계 | 2단계 (Phase1 → Pseudo 생성 → Phase2) |
| 자동 채택 | - | Phase1 vs Phase2 자동 비교 후 최선 선택 |

### Pseudo Label 튜닝 가이드
- `PSEUDO_THRESHOLD=0.05` → 신뢰도 95% 이상만 (보수적, 샘플 적음)
- `PSEUDO_THRESHOLD=0.10` → 신뢰도 90% 이상 (균형적, 권장)
- `PSEUDO_THRESHOLD=0.15` → 신뢰도 85% 이상 (공격적, 노이즈 가능)

### Focal Loss 튜닝 가이드
- unstable 비율 확인 후 `FOCAL_ALPHA` 조정
- `FOCAL_GAMMA=0` → 일반 BCE와 동일 (비교 기준)
- `FOCAL_GAMMA=1.5` → 권장값
- `FOCAL_GAMMA=2.0` → 강한 집중 (데이터가 충분할 때)